# Bank Loan Portfolio — Data Quality Audit

This diagnostic notebook tests whether the supplied extract is safe for portfolio-risk analysis. Warnings are retained as evidence; rows are never silently deleted.


## tl;dr

- Core grain, status, amount, rate, and issue-date guardrails are expected to pass.
- Cross-field payment and credit-pull dates are not governed and must not support timing or maturity claims.
- `emp_title` missingness and high-income outliers are retained and documented.


## Context & Methods

### Key Assumptions
- Source grain is one row per loan ID.
- `issue_date` is the only date used for origination trends.
- Payment/credit-pull chronology remains a source-validation issue, not a cleaning target.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

from IPython.display import display

root = Path.cwd().resolve()
for candidate in [root, *root.parents]:
    if (candidate / 'data' / 'financial_loan.csv').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('Cannot find data/financial_loan.csv')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_quality import run_full_audit
DATA_PATH = PROJECT_ROOT / 'data' / 'financial_loan.csv'
raw_loans = pd.read_csv(DATA_PATH)
print(f'Loaded {len(raw_loans):,} rows × {raw_loans.shape[1]} columns')


## Data


In [ ]:
display(raw_loans.head(10))
display(raw_loans.dtypes.rename('dtype').to_frame())


## Results

### 1. Run the governed audit


In [ ]:
audit = run_full_audit(raw_loans)
display(audit['dq_summary'])
print(f"PASS={audit['n_pass']} | WARN={audit['n_warn']} | FAIL={audit['n_fail']}")


### 2. Review missingness and normalization


In [ ]:
display(audit['missing_values'])
display(audit['string_normalization'])


### 3. Review temporal contradictions


In [ ]:
display(audit['temporal_consistency'].style.format({'affected_rate': '{:.2%}'}))


### 4. Review outliers and status coverage


In [ ]:
display(audit['outlier_summary'])
display(audit['status_distribution'])


## Takeaways


In [ ]:
blocking = audit['n_fail'] > 0
print('BLOCK KPI PUBLICATION' if blocking else 'CORE KPI GUARDRAILS PASS')
print('Temporal payment/credit-pull fields remain quarantined until source semantics are verified.')
print('No source rows were deleted or silently imputed.')
